# 06 — Gold Layer: Star Schema Dimensions

Creates and persists static dimension tables for the Star Schema to `tlc_gold` in MongoDB.

Dimensions:
- `dim_vendor` — taxi vendor IDs
- `dim_rate_code` — rate codes and descriptions
- `dim_payment_type` — payment methods
- `dim_vehicle` — vehicle type categories
- `dim_zone` — TLC taxi zone lookup (265 zones)
- `dim_date` — date spine from 2024-01-01 to **2026-12-31** with hour, day_name, is_weekend fields for BI dashboards

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work')

from src.spark_utils import get_spark, write_mongo
from core.config.settings import settings
from src import paths
import pyspark.sql.functions as F
import pyspark.sql.types as T

print("Imports OK.")

In [ ]:
spark = get_spark("TLC_Gold_Dimensions")

## dim_vendor

In [ ]:
vendor_data = [(1, "Creative Mobile Technologies"), (2, "VeriFone")]
dim_vendor = spark.createDataFrame(vendor_data, ["id", "name"])
write_mongo(dim_vendor, settings.MONGO_DB_GOLD, "dim_vendor", mode="overwrite")
print(f"dim_vendor: {dim_vendor.count()} rows")

## dim_rate_code

In [ ]:
rate_code_data = [
    (1, "Standard rate"),
    (2, "JFK"),
    (3, "Newark"),
    (4, "Nassau or Westchester"),
    (5, "Negotiated fare"),
    (6, "Group ride"),
]
dim_rate_code = spark.createDataFrame(rate_code_data, ["id", "description"])
write_mongo(dim_rate_code, settings.MONGO_DB_GOLD, "dim_rate_code", mode="overwrite")
print(f"dim_rate_code: {dim_rate_code.count()} rows")

## dim_payment_type

In [ ]:
payment_type_data = [
    ("Credit Card", "Credit Card"),
    ("Cash",        "Cash"),
    ("No Charge",   "No Charge"),
    ("Dispute",     "Dispute"),
    ("Unknown",     "Unknown"),
    ("Voided Trip", "Voided Trip"),
]
dim_payment_type = spark.createDataFrame(payment_type_data, ["id", "name"])
write_mongo(dim_payment_type, settings.MONGO_DB_GOLD, "dim_payment_type", mode="overwrite")
print(f"dim_payment_type: {dim_payment_type.count()} rows")

## dim_vehicle

In [ ]:
vehicle_data = [
    ("yellow", "Yellow Taxi"),
    ("green",  "Green Taxi"),
    ("fhv",    "For-Hire Vehicle"),
    ("hvfhv",  "High Volume FHV"),
]
dim_vehicle = spark.createDataFrame(vehicle_data, ["id", "name"])
write_mongo(dim_vehicle, settings.MONGO_DB_GOLD, "dim_vehicle", mode="overwrite")
print(f"dim_vehicle: {dim_vehicle.count()} rows")

## dim_zone

In [ ]:
dim_zone = (
    spark.read.csv(str(paths.TAXI_ZONE_LOOKUP), header=True, inferSchema=True)
    .select(
        F.col("LocationID").alias("zone_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone_name"),
        F.col("service_zone"),
    )
)
write_mongo(dim_zone, settings.MONGO_DB_GOLD, "dim_zone", mode="overwrite")
print(f"dim_zone: {dim_zone.count()} rows (TLC zones 1-265)")

## dim_date

Date spine from **2024-01-01** to **2026-12-31** with extended fields:
- `hour` (0-23) — required for SARIMA hourly seasonality analysis
- `day_name` (Monday…Sunday) — for descriptive dashboards
- `is_weekend` — boolean flag for diagnostic filtering
- `quarter` — for BI aggregations

In [ ]:
# Generate one row per day (2024-01-01 → 2026-12-31)
dim_date = (
    spark.createDataFrame([(1,)], ["dummy"])
    .withColumn(
        "date",
        F.explode(
            F.sequence(
                F.to_date(F.lit("2024-01-01")),
                F.to_date(F.lit("2026-12-31")),
            )
        ),
    )
    .drop("dummy")
    .select(
        F.date_format(F.col("date"), "yyyyMMdd").cast("int").alias("date_id"),
        F.col("date"),
        F.year(F.col("date")).alias("year"),
        F.quarter(F.col("date")).alias("quarter"),
        F.month(F.col("date")).alias("month"),
        F.date_format(F.col("date"), "MMMM").alias("month_name"),
        F.dayofmonth(F.col("date")).alias("day"),
        F.dayofweek(F.col("date")).alias("day_of_week"),   # 1=Sun, 7=Sat (Spark default)
        F.date_format(F.col("date"), "EEEE").alias("day_name"),  # Monday…Sunday
        # is_weekend: Saturday (dow=7) or Sunday (dow=1)
        F.when(F.dayofweek(F.col("date")).isin(1, 7), True)
         .otherwise(False)
         .alias("is_weekend"),
        F.dayofyear(F.col("date")).alias("day_of_year"),
        F.weekofyear(F.col("date")).alias("week_of_year"),
    )
)

n_dates = dim_date.count()
write_mongo(dim_date, settings.MONGO_DB_GOLD, "dim_date", mode="overwrite")
print(f"dim_date: {n_dates} rows (2024-01-01 → 2026-12-31)")
dim_date.show(5, truncate=False)

## dim_hour

Separate 24-row dimension for hour-of-day analysis (used by SARIMA seasonality s=24).

In [ ]:
hour_data = [
    (h, f"{h:02d}:00", "Night" if h < 6 else "Morning" if h < 12 else "Afternoon" if h < 18 else "Evening")
    for h in range(24)
]
dim_hour = spark.createDataFrame(hour_data, ["hour", "hour_label", "time_of_day"])
write_mongo(dim_hour, settings.MONGO_DB_GOLD, "dim_hour", mode="overwrite")
print(f"dim_hour: {dim_hour.count()} rows")
dim_hour.show(truncate=False)